# Mi proyecto · De la teoría a la práctica (Sesiones 1 a 3)


---


**Plantilla espejo del caso guiado Olist** · Módulo: Preparación, Recopilación y Análisis Exploratorio de Datos · MIADAS – UMSA

#### **Grupo: 1** - Integrantes:

*   Denisse Marin Ayala
*   Dardo Gamarra
*   Nelson De La Quintana
*   Mauricio De La Quintana

---

**Conjunto de datos:** The Movies Dataset

**Fuente y URL:** Kaggle (`https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset`)

Esta plantilla tiene **las mismas secciones** que el caso guiado. Repite cada paso con **tu conjunto de datos**. Este notebook evolucionará en las próximas sesiones con la limpieza, el análisis exploratorio, la normalización y la calidad.

### Requisitos mínimos de tu conjunto de datos
- [X] Al menos **1 000 registros**
- [X] Al menos **8 variables** que mezclen escalas (nominal, ordinal, de razón)
- [X] Al menos **una variable de fecha**
- [X] **Problemas de calidad** reales (faltantes, duplicados o inconsistencias)
- [X] Idealmente, **dos tablas o fuentes** que se puedan integrar
- [X] **Licencia** que permita su uso académico


---

**Nombre del proyecto:**
### Análisis de rentabilidad y recepción del público en la industria cinematográfica

**Pregunta de Negocio** ¿Qué relación existe entre el presupuesto invertido, el género y la duración de una película con su recaudación en taquilla y la calificación promedio otorgada por el público?

---

### Paso 0 · Cargar los Datos  · (Desde Kaggle)

In [195]:
import math
import os
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

movies   = pd.read_csv(os.path.join(path, "movies_metadata.csv"), low_memory=False)
credits  = pd.read_csv(os.path.join(path, "credits.csv"), low_memory=False)
keywords = pd.read_csv(os.path.join(path, "keywords.csv"), low_memory=False)
links    = pd.read_csv(os.path.join(path, "links.csv"), low_memory=False)

print("Películas: ", movies.shape)
print("Créditos:  ", credits.shape)
print("Palabras:  ", keywords.shape)
print("Enlaces:   ", links.shape)

Using Colab cache for faster access to the 'the-movies-dataset' dataset.
Películas:  (45466, 24)
Créditos:   (45476, 3)
Palabras:   (46419, 2)
Enlaces:    (45843, 3)


---
# S1 · Tipos de datos y escalas (Sesión 1)
### S1.1 · Tipos de dato
¿Qué columnas se leyeron con un tipo incorrecto (fechas como texto, códigos como números)?

In [196]:
print("----- 1. TABLA MOVIES -----")
movies.info()

----- 1. TABLA MOVIES -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           453

### S1.2 · Corrige los tipos
Convierte las fechas con `pd.to_datetime` y los códigos (identificadores, códigos postales) a texto con `astype(str)`.

In [197]:
# Convertimos la fecha de estreno a datetime
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")

# Aseguramos que el identificador 'id' sea texto
movies["id"] = movies["id"].astype(str)

print("\nFechas convertidas con éxito:")
print(movies[["release_date"]].dtypes)
print("Primer lanzamiento:", movies["release_date"].min())
print("Último lanzamiento:", movies["release_date"].max())

# Limpiamos y convertimos variables financieras y de razón a numérico
movies["budget"] = pd.to_numeric(movies["budget"], errors="coerce")
movies["popularity"] = pd.to_numeric(movies["popularity"], errors="coerce")

print("Tipos de datos corregidos y limpios.")


Fechas convertidas con éxito:
release_date    datetime64[ns]
dtype: object
Primer lanzamiento: 1874-12-09 00:00:00
Último lanzamiento: 2020-12-16 00:00:00
Tipos de datos corregidos y limpios.


**Resultado esperado:** la columna paso a `datetime64`; el primer lanzamiento es del **09 de diciembre de 1874** y el último del **16 de diciembre de 2020**.

####**Observaciones en los tipos de datos:**
Al revisar movies.info(), detectamos 3 problemas típicos de datos sin tratar:


---
> 1. **Fechas como texto:** `release_date` se leyó como objeto y requirió conversión a `datetime`.
> 2. **Variables financieras corruptas:** `budget` se leía como texto debido a anomalías de formato en filas específicas, por lo que fue necesario forzar su conversión a tipo numérico.
> 3. **Identificadores:** `id` se mantiene estrictamente como texto (`str`) por ser una variable nominal.



(Inspección de las 3 tablas restantes)

In [198]:
print("----- 2. TABLA CREDITS -----")
credits.info()

print("\n---- 3. TABLA KEYWORDS -----")
keywords.info()

print("\n----- 4. TABLA LINKS -----")
links.info()


----- 2. TABLA CREDITS -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   crew    45476 non-null  object
 2   id      45476 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.0+ MB

---- 3. TABLA KEYWORDS -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46419 entries, 0 to 46418
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        46419 non-null  int64 
 1   keywords  46419 non-null  object
dtypes: int64(1), object(1)
memory usage: 725.4+ KB

----- 4. TABLA LINKS -----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45843 entries, 0 to 45842
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  45843 non-null  int64  
 1   imdbId   45843 non-null  int64  
 2 

 (Corrección de las 3 tablas restantes)

In [199]:
# Estandarizamos todos los identificadores (variables nominales) a texto (str)
credits["id"] = credits["id"].astype(str)
keywords["id"] = keywords["id"].astype(str)

links["movieId"] = links["movieId"].astype(str)
links["imdbId"] = links["imdbId"].astype(str)
links["tmdbId"] = links["tmdbId"].astype("Int64").astype(str)  # Evita decimales (.0) por los valores nulos

print("Tipos corregidos en credits, keywords y links:")
print("credits['id']  :", credits["id"].dtype)
print("keywords['id'] :", keywords["id"].dtype)
print("links dtypes   :\n", links.dtypes)


Tipos corregidos en credits, keywords y links:
credits['id']  : object
keywords['id'] : object
links dtypes   :
 movieId    object
imdbId     object
tmdbId     object
dtype: object


#### **Observaciones en los tipos de datos (tablas complementarias):**

---
> 1. **Identificadores numéricos en `keywords` y `links`:** Los códigos `id`, `movieId` e `imdbId` se leyeron como enteros (`int64`), mientras que `tmdbId` se leyó como decimal (`float64`) debido a la presencia de 219 valores nulos. Al ser variables nominales de identificación, se convirtieron a texto (`str`).
> 2. **Incompatibilidad de claves para integración:** Mientras `id` en `movies` y `credits` estaban como texto (`object`), en `keywords` era entero y en `links` decimal. Dicho esto, estandarizamos todas las claves a `str` evitando asi los errores de incompatibilidad al momento de cruzar las tablas.


### S1.3 · Escala de cada variable

---

Completa la tabla con tus variables (mínimo 8):

| Variable | Escala (nominal / ordinal / intervalo / razón) | Cálculos válidos |
|---|---|---|
|`original_language`(Idioma Original)|Nominal|Frecuencias, moda|
|`genres`(Generos)|Nominal|Frecuencias, moda|
|`vote_average`(Promedio de Votos)|Ordinal|Mediana, percentiles, media|
|`release_date`(Fecha de Estreno)|Intervalo|Diferencias entre fechas, rangos|
|`budget`, `revenue`(presupuesto)|Razón|Todos, incluidas razones y proporciones|
|`runtime(Duración)`|Razón|Todos, incluidas razones y proporciones|
|`popularity`(Popularidad)|Razón|Todos, incluidas razones y proporciones|
|`id`(identificador)|Nominal|Conteo de registros únicos|

### **Analisis por Escala de cada Variable**


---
#### Variables con Escala Nominal:


In [200]:
print("--- Frecuencias de idioma de Origen de las Peliculas (Top 5) ---")
print(movies["original_language"].value_counts().head(5))

--- Frecuencias de idioma de Origen de las Peliculas (Top 5) ---
original_language
en    32269
fr     2438
it     1529
ja     1350
de     1080
Name: count, dtype: int64


**Resultado Observado**. en (inglés) 32.269, fr (francés) 2.438, it
(italiano) 1.529, ja (japonés) 1.350, de (alemán) 1.080.


---



**Interpretación del Resultado**.

**• Variable nominal (original_language):** Alrededor del 71 % de las películas registradas están en inglés (en), con respecto al catálogo total (32.269 películas de un total de 45.466 registros). Esto indica un claro predominio de producciones angloparlantes frente al resto de idiomas.

In [201]:
print("--- Frecuencias de Géneros de las Peliculas (Top 5) ---")
generos = movies["genres"].str.findall(r"'name': '([^']+)'").str.join(", ").replace("", "Sin género")
print(generos.value_counts().head(5))


--- Frecuencias de Géneros de las Peliculas (Top 5) ---
genres
Drama             5000
Comedy            3621
Documentary       2723
Sin género        2442
Drama, Romance    1301
Name: count, dtype: int64


**Resultado Observado**. Drama 5.000, Comedy (Comedia) 3.621, Documentary (Documental) 2.723, Sin género 2.442, Drama, Romance 1.301.

---

**Interpretación del Resultado**.

**• Variable nominal (genres):** Alrededor del 11 % de las películas registradas pertenecen exclusivamente al género Drama, con respecto al catálogo total (5.000 películas de un total de 45.466 registros). Esto indica un predominio de producciones dramáticas frente al resto de categorías y combinaciones, destacando además un 5,4 % (2.442 registros) que no tienen género asignado en la base de datos.

In [202]:
print("--- Unicidad de identificador (id) ---")
print(f"Registros totales : {len(movies)}")
print(f"IDs únicos        : {movies['id'].nunique()}")
print(f"IDs duplicados    : {len(movies) - movies['id'].nunique()}")

--- Unicidad de identificador (id) ---
Registros totales : 45466
IDs únicos        : 45436
IDs duplicados    : 30


**Resultado Observado**. Registros totales: 45.466 , IDs únicos: 45.436 , IDs duplicados: 30.

---

**Interpretación del Resultado**.

**• Variable nominal (id):** Al tratarse de un identificador único para cada película, no se resume mediante moda sino evaluando su unicidad. Se encontraron 45.436 identificadores únicos dentro de un total de 45.466 registros (99,93 % de unicidad), lo que revela la presencia de 30 filas duplicadas en la tabla.


---
#### Variables con Escala Razón:

In [203]:
print("\n--- Estadísticos de variables de razón ---")
movies[["budget", "revenue", "runtime", "popularity"]].describe().round(2)


--- Estadísticos de variables de razón ---


,budget,revenue,runtime,popularity
count,4.546300e+04,4.546000e+04,45203.00,45460.00
mean,4.224579e+06,1.120935e+07,94.13,2.92
std,1.742413e+07,6.433225e+07,38.41,6.01
min,0.000000e+00,0.000000e+00,0.00,0.00
25%,0.000000e+00,0.000000e+00,85.00,0.39
50%,0.000000e+00,0.000000e+00,95.00,1.13
75%,0.000000e+00,0.000000e+00,107.00,3.68
max,3.800000e+08,2.787965e+09,1256.00,547.49


**Interpretación de Resultados:**

---

**Variables de razón (budget, revenue, runtime, popularity):**

 **• Presupuesto (budget) e ingresos (revenue):** En budget (media de 4,22 millones) y revenue (media de 11,21 millones), la mediana (50 %) y hasta el percentil 75 % son 0,00, mientras que los máximos llegan a 380 millones y 2.788 millones. Esto evidencia una gran cantidad de películas sin datos financieros reportados (registrados como 0) y pocos éxitos de taquilla que elevan drásticamente el promedio.

  **• Duración (runtime):** En runtime, la media (94,13 min) y la mediana (95,00 min) son casi iguales, indicando que la duración típica ronda la hora y media, aunque el máximo de 1.256 min y el mínimo de 0 min advierten valores atípicos.

  **• Popularidad (popularity):** Presenta una mediana (50%) de 1,13 y un percentil 75% de 3,68, lo que indica que la gran mayoría del catálogo tiene una popularidad baja o moderada. Sin embargo, la media sube a 2,92 (con una desviación estándar de 6,01) debido a películas virales o muy famosas cuyo índice se dispara hasta un máximo atípico de 547,49.


In [204]:
# Filtrar solo películas con presupuesto e ingresos mayores a 0
movies_finanzas = movies[(movies["budget"] > 0) & (movies["revenue"] > 0)]

print("--- Estadísticos reales (sin valores en cero) ---")
print(movies_finanzas[["budget", "revenue", "runtime", "popularity"]].describe().round(2))

--- Estadísticos reales (sin valores en cero) ---
             budget       revenue  runtime  popularity
count  5.381000e+03  5.381000e+03  5380.00     5381.00
mean   3.109480e+07  9.031812e+07   109.83        9.77
std    4.016263e+07  1.661423e+08    22.04       13.82
min    1.000000e+00  1.000000e+00     0.00        0.00
25%    5.037000e+06  7.011317e+06    95.00        5.44
50%    1.700000e+07  2.991874e+07   106.00        8.39
75%    4.000000e+07  9.996575e+07   120.00       11.72
max    3.800000e+08  2.787965e+09   338.00      547.49


**Interpretación de Estadísticos Reales (sin valores en cero):**

---

Al filtrar únicamente las películas que reportan presupuesto e ingresos mayores a 0, la muestra se reduce a 5.381 películas (aproximadamente el 12 % del catálogo original). En este subconjunto real, el presupuesto medio (mean) sube a 31,09 millones y la mediana (50%) a 17,00 millones. Por su parte, los ingresos medios alcanzan los 90,32 millones, frente a una mediana de 29,92 millones. En ambos casos, la media sigue siendo mucho mayor que la mediana porque existen éxitos de taquilla extremos (con recaudaciones máximas de hasta 2.788 millones) que elevan el promedio general. Asimismo, al limpiar los ceros financieros, la popularidad (popularity) aumenta considerablemente su nivel típico, pasando de una mediana de 1,13 a 8,39 y una media de 9,77, impulsada por los grandes estrenos comerciales cuyo índice alcanza hasta 547,49 (mientras que la duración típica en runtime se estabiliza en una mediana de 106,00 minutos, eliminando además el valor extremo anterior de 1.256 minutos).



---
#### Variables con Escala Ordinal / Intervalo:

In [205]:
print("--- Distribución de frecuencias ordenada (vote_average) ---")
print(movies["vote_average"].round().value_counts().sort_index())

print("\n--- Comparación Media vs. Mediana ---")
print(f"Media  : {movies['vote_average'].mean():.2f}")
print(f"Mediana: {movies['vote_average'].median():.2f}")


--- Distribución de frecuencias ordenada (vote_average) ---
vote_average
0.0      3011
1.0       129
2.0       413
3.0       817
4.0      3506
5.0      7343
6.0     15675
7.0     10229
8.0      3831
9.0       296
10.0      210
Name: count, dtype: int64

--- Comparación Media vs. Mediana ---
Media  : 5.62
Mediana: 6.00


**Resultado Observado**. Media: 5,62 · Mediana: 6,00. En la distribución ordenada de puntajes (de 0 a 10), la mayor concentración de películas se ubica en las calificaciones de 6 y 7 puntos, mientras que aparece un grupo considerable en 0 puntos (películas sin calificación registrada).

---

**Interpretación del Resultado**.

• Variable ordinal (vote_average): Según la escala de Stevens, aunque se puede calcular la media (5,62), el estadístico riguroso y más adecuado para una variable ordinal es la mediana (6,00) junto con su distribución de frecuencias. Esto se debe a que las calificaciones representan un orden de preferencia subjetivo donde la distancia entre un puntaje y otro no es matemáticamente uniforme, y además la mediana evita que las películas sin votos (registradas con 0) distorsionen el valor central del catálogo hacia abajo.


In [206]:
fechas = pd.to_datetime(movies["release_date"], errors="coerce")

print("--- Estadísticos de variable de intervalo (release_date) ---")
print(f"Registros válidos  : {fechas.count():,}")
print(f"Registros nulos    : {fechas.isna().sum():,}")
print(f"Fecha más antigua  : {fechas.min().date()}")
print(f"Fecha más reciente : {fechas.max().date()}")
print(f"Rango total        : {fechas.max().year - fechas.min().year} años")
print(f"Año mediano (50 %) : {int(fechas.dt.year.median())}")
print(f"Año más frecuente  : {int(fechas.dt.year.mode()[0])} ({(fechas.dt.year == fechas.dt.year.mode()[0]).sum():,} películas)")

--- Estadísticos de variable de intervalo (release_date) ---
Registros válidos  : 45,376
Registros nulos    : 90
Fecha más antigua  : 1874-12-09
Fecha más reciente : 2020-12-16
Rango total        : 146 años
Año mediano (50 %) : 2001
Año más frecuente  : 2014 (1,974 películas)


**Resultado Observado**. Registros válidos: 45 376 · Registros nulos: 90 · Fecha más antigua: 1874-12-09 · Fecha más reciente: 2020-12-16 · Rango total: 146 años · Año mediano: 2001 · Año más frecuente (moda): 2014 (1 974 películas).

---

**Interpretación del Resultado**.

• Variable de intervalo (release_date): Al tratarse de una escala de intervalo según Stevens, las operaciones válidas son el cálculo de rangos, diferencias temporales y medidas de tendencia central como la mediana y la moda. El catálogo abarca un rango histórico de 146 años (desde diciembre de 1874 hasta diciembre de 2020). Sin embargo, el año mediano se ubica en 2001 y el pico de producción alcanza su moda en 2014 (con 1 974 estrenos), lo que demuestra que la mitad del catálogo se concentra en las últimas dos décadas. Además, se detectan 90 registros sin fecha válida.


---
# S2 · Muestreo (Sesión 2)
¿Qué subconjunto necesitarías revisar a mano o encuestar? Calcula el tamaño de muestra y compara una muestra aleatoria con una estratificada.

In [207]:
def tamano_muestra(N, e=0.05, z=1.96, p=0.5):
    n0 = z**2 * p * (1 - p) / e**2
    return math.ceil(n0 / (1 + (n0 - 1) / N))
# Cochran con el total de filas de la tabla movies
n = tamano_muestra(len(movies))
print(f"Total de peliculas (N): {len(movies)} | Tamaño de muestra (n): {n}")


Total de peliculas (N): 45466 | Tamaño de muestra (n): 381


In [208]:
# Primero preparamos el estrato agrupando idiomas minoritarios en "Otros"
top_5 = movies["original_language"].value_counts().head(5).index
movies["estrato_idioma"] = movies["original_language"].where(movies["original_language"].isin(top_5), "Otros")

variable_estrato = "estrato_idioma"
aleatoria = movies.sample(n=n, random_state=42)
estratificada = movies.groupby(variable_estrato, group_keys=False).sample(frac=n / len(movies), random_state=42)

comparacion = pd.DataFrame({
    "Población": movies[variable_estrato].value_counts(normalize=True),
    "Aleatoria": aleatoria[variable_estrato].value_counts(normalize=True),
    "Estratificada": estratificada[variable_estrato].value_counts(normalize=True),
}).sort_index().round(3)

comparacion


,Población,Aleatoria,Estratificada
estrato_idioma,,,
Otros,0.150,0.144,0.150
de,0.024,0.016,0.024
en,0.710,0.735,0.711
fr,0.054,0.055,0.053
it,0.034,0.021,0.034
ja,0.030,0.029,0.029


**Resultado observado:**

• Idioma dominante (en — inglés): Población: 0,710 · Aleatoria: 0,735 · Estratificada: 0,711.

• Idioma minoritario (it — italiano): Población: 0,034 · Aleatoria: 0,021 · Estratificada: 0,034.

---

**Aplicación práctica y conclusión:**

Revisar manualmente las 45.466 películas para controlar la calidad de sus subtítulos y sinopsis tomaría demasiado tiempo y presupuesto. Con la fórmula de Cochran reducimos esa tarea a una muestra manejable de 381 títulos (95 % de confianza, 5 % de error), agrupando los idiomas minoritarios en "Otros" para mantener la base completa. Los resultados demuestran que la muestra estratificada refleja fielmente las proporciones originales (0,711 para inglés y 0,034 para italiano), mientras que la selección aleatoria altera los pesos por azar, sobreestimando el inglés (0,735) y dejando fuera parte del cine italiano (0,021). Por ello, cuando una categoría domina ampliamente sobre las demás, el muestreo estratificado garantiza una auditoría mucho más justa y precisa.


---
# S3 · Fuentes e integración (Sesión 3)

---

#### S3.1 · El modelo de datos de The Movies Dataset
```
credits ──(id)── movies ──(id)── keywords
                    │
               (id / tmdbId)
                    │
                  links

```
• id une movies (metadatos financieros, género, duración y calificación) con credits (elenco y equipo técnico) y con keywords (palabras clave temáticas).
• id (en movies) equivale a tmdbId (en links), permitiendo vincular el catálogo con identificadores externos de IMDb (imdbId) y MovieLens (movieId).


#### S3.2 · ¿Cómo integramos los metadatos con el elenco y equipo técnico?

> El elenco (cast) y el equipo de producción (crew) están en otra tabla (credits). Las unimos con merge, como un JOIN de SQL:


In [209]:
# Integración
peliculas_creditos = movies.merge(credits, on="id", how="left")

print("Dimensiones de la tabla integrada resultante:", peliculas_creditos.shape)
peliculas_creditos[["title", "cast", "crew"]].head(3)


Dimensiones de la tabla integrada resultante: (45542, 27)


,title,cast,crew
0,Toy Story,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,Jumanji,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,Grumpier Old Men,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."


**Resultado Observado tras la integración:**

---

>  Dimensión tras unir movies + credits: (45.542, 27).

>  Al realizar la unión mediante la clave id, las filas aumentan ligeramente de 45.466 a 45.542 (76 filas extra) y las columnas pasan de 24 a 27. Esto confirma la existencia de registros con id duplicados en ambas tablas originales que generan multiplicación de filas en el cruce, un hallazgo clave para depurar mas adelante en la fase de limpieza.

* Observación sobre la estructura de datos: Se identifica que las columnas incorporadas (cast y crew) almacenan listas de diccionarios en formato JSON anidado dentro de cada celda. Por ello, su extracción y desglose (por ejemplo, para obtener nombres de directores o actores principales) se debera abordar en las próximas etapas de transformación y limpieza de datos.



#### S3.3 · La pregunta de negocio: ¿Qué relación existe entre el presupuesto invertido, el género y la duración con la recaudación en taquilla y la calificación del público?

---

>Una película se considera rentable si su recaudación en taquilla supera su presupuesto invertido (revenue > budget). Para esta primera aproximación exploratoria, filtramos sobre la tabla principal (`movies`, evitando los duplicados generados en el cruce) únicamente las películas con datos financieros válidos (`budget > 1000` y `revenue > 1000`).

In [210]:
# Filtramos películas que tengan registros financieros reales (> 1000 USD)
validas = movies[(movies["budget"] > 1000) & (movies["revenue"] > 1000)].copy()
validas["rentable"] = validas["revenue"] > validas["budget"]

print("Películas válidas analizadas:", len(validas))
print("Fueron rentables:", validas["rentable"].sum())
print("Porcentaje rentable:", round(validas["rentable"].mean() * 100, 1), "%")
print("\nCalificación promedio según rentabilidad:")
print(validas.groupby("rentable")["vote_average"].mean().round(2))


Películas válidas analizadas: 5312
Fueron rentables: 3725
Porcentaje rentable: 70.1 %

Calificación promedio según rentabilidad:
rentable
False    5.94
True     6.42
Name: vote_average, dtype: float64


> **Primer hallazgo de negocio:**
> Resultado observado. 5 312 películas analizadas · 3 725 rentables · 70,1 % (calificación promedio: 6,42 en rentables frente a 5,94 en no rentables).

---

> De las 5.312 películas con presupuestos y recaudaciones reales reportadas (> 1 000 USD), el 70,1 % logra superar su costo de producción en taquilla. Asimismo, las producciones rentables obtienen una calificación promedio del público superior (6,42 frente a 5,94 de las no rentables), evidenciando de forma preliminar una relación positiva entre el éxito financiero y la valoración de la audiencia.

---
# Cierre · Informe de recolección inicial (CRISP-DM, fase 2)

| Elemento | Mi conjunto de datos |
|---|---|
| **Fuente** |Kaggle · *The Movies Dataset* (recopilado de metadatos de TMDb y GroupLens)|
| **Tipo de fuente** (interna / externa · primaria / secundaria) |Externa · datos secundarios · recopilados de bases de datos cinematográficas públicas|
| **Formato y acceso** (CSV, Excel, API… · URL) |CSV · descarga directa vía API (kagglehub)|
| **Tablas y número de registros** |movies_metadata (45 466) · credits (45 476) · keywords (46 419) · links (45 843)|
| **Periodo que cubre** |9 de diciembre de 1874 a 16 de diciembre de 2020|
| **Problemas encontrados** |Fechas leídas como texto y 90 nulos en release_date · más del 87 % de registros con budget o revenue en cero o valores simbólicos · 30 IDs duplicados en movies (que generan 76 filas extra al cruzar con credits) · tipos incompatibles en la clave id · 2 442 películas sin género ([]) · columnas almacenadas en formato JSON (genres, cast, keywords)|
| **Primer hallazgo o pregunta de negocio** |El 70,1 % de las películas con finanzas válidas (3.725 de 5.312) supera su presupuesto en taquilla, con 6,42 puntos de calificación promedio frente a 5,94 de las no rentables|